# SWED / SWBD methodology schema — sandbox

Interactive version of `schema_explanation_era5.py`: load the region once, then change the year and thresholds and re-plot.
All computation and plotting functions come from the script, so any change to the figure style should be made there.

In [ ]:
# --- SWED threshold definition --------------------------------------------
# "global"  : one P(SWED_Q) threshold of non-zero wcf/scf over the whole period
# "seasonal": one threshold per calendar day, from all years' days within a
#             centred rolling window of SEASONAL_WINDOW days (+-15 d = a rolling month)
SWED_METHOD = "seasonal"   # "global" or "seasonal"
SEASONAL_WINDOW = 31

In [1]:
import netCDF4  # noqa: F401  -- must be imported before anything pulling in rasterio/GDAL
import importlib
import matplotlib.pyplot as plt
import schema_explanation_era5 as sch

importlib.reload(sch)  # re-run this cell after editing the script
%matplotlib inline
plt.rcParams["figure.dpi"] = 200

## 1. Load the region (once)

In [2]:
REGION = "Pakistan"

wcf, scf, tas, poly_idx = sch.load_region_series(REGION)
print(f"{REGION}: poly_idx={poly_idx}, {str(wcf.time.values[0])[:10]} -> {str(wcf.time.values[-1])[:10]}")

Pakistan: poly_idx=91, 1995-01-01 -> 2014-12-31


## 2. Parameters to play with

In [ ]:
SWED_Q = 0.10   # quantile of non-zero wcf/scf (SWED threshold)
SWBD_Q = 0.99   # quantile of residual load (SWBD threshold)
TOT_RE = 0.5    # renewable penetration: rl = demand - TOT_RE * supply

wthr, sthr, swed = sch.compute_swed(wcf, scf, q=SWED_Q,
                                     method=SWED_METHOD, window=SEASONAL_WINDOW)
demand, supply, rl, rl_thr, swbd, solar_share = sch.compute_swbd(
    wcf, scf, tas, poly_idx, tot_re=TOT_RE, q=SWBD_Q)

print(f"solar share = {solar_share:.2f}")
def _fmt(thr):  # float for "global", min-max range over the year for "seasonal"
    return f"{thr:.4f}" if isinstance(thr, float) else f"{float(thr.min()):.4f}-{float(thr.max()):.4f}"

print(f"SWED method = {SWED_METHOD}" + (f" ({SEASONAL_WINDOW}-day window)" if SWED_METHOD == "seasonal" else ""))
print(f"wind thr = {_fmt(wthr)} | solar thr = {_fmt(sthr)} | RL thr = {rl_thr:.4f}")

## 3. Event days per year — helps pick a year

In [9]:
import pandas as pd

common = swed & swbd  # days that are both a SWED and a SWBD day
print(f"Total common days: {int(common.sum())}")

counts = pd.DataFrame({
    "SWED days": swed.groupby("time.year").sum().to_series(),
    "SWBD days": swbd.groupby("time.year").sum().to_series(),
    "common days": common.groupby("time.year").sum().to_series(),
})
counts["total"] = counts["SWED days"] + counts["SWBD days"]
counts.sort_values("total", ascending=False)

Total common days: 1


,SWED days,SWBD days,common days,total
year,,,,
2013,16,5,0,21
1999,12,7,0,19
2014,8,7,0,15
2003,7,7,0,14
2007,4,8,0,12
2010,4,7,0,11
1998,5,6,0,11
2008,7,3,0,10
2011,2,7,0,9


### Global vs seasonal SWED — how many days coincide with SWBD?

In [ ]:
# Global vs seasonal SWED definition, same SWBD -- totals over the whole period
rows = {}
for m in ("global", "seasonal"):
    _, _, sw = sch.compute_swed(wcf, scf, q=SWED_Q, method=m, window=SEASONAL_WINDOW)
    rows[m] = {"SWED days": int(sw.sum()),
               "SWBD days": int(swbd.sum()),
               "common days": int((sw & swbd).sum())}
pd.DataFrame(rows).T

## 4. Plot one year

In [ ]:
YEAR = int(counts["total"].idxmax())  # or set e.g. YEAR = 2003

fig = sch.plot_swed_swbd_schema(
    wcf, scf, wthr, sthr, swed, demand, supply, rl, rl_thr, swbd, YEAR,
    tot_re=TOT_RE, swed_q=SWED_Q, swbd_q=SWBD_Q, swed_method=SWED_METHOD)
plt.show()

## 5. Compare several years quickly

In [ ]:
for y in counts.sort_values("total", ascending=False).index[:3]:
    fig = sch.plot_swed_swbd_schema(
        wcf, scf, wthr, sthr, swed, demand, supply, rl, rl_thr, swbd, int(y),
        tot_re=TOT_RE, swed_q=SWED_Q, swbd_q=SWBD_Q, swed_method=SWED_METHOD)
    plt.show()

## 6. Save the chosen version

In [ ]:
import os

os.makedirs(sch.OUT_DIR, exist_ok=True)
out_path = os.path.join(sch.OUT_DIR, f"suppfig1_swed_swbd_schema_{REGION.lower()}_{YEAR}_swed-{SWED_METHOD}.png")
fig = sch.plot_swed_swbd_schema(
    wcf, scf, wthr, sthr, swed, demand, supply, rl, rl_thr, swbd, YEAR,
    tot_re=TOT_RE, swed_q=SWED_Q, swbd_q=SWBD_Q, swed_method=SWED_METHOD)
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print("Saved", out_path)